# TC3-82 – Levantar e integrar bases socioeconômicas e financeiras municipais

**Responsável:** Mauricio Cruz

**Sprint:** Fase 2

**Jira:** TC3-82

## Objetivo

Levantar, baixar, documentar e integrar as novas bases socioeconômicas e financeiras municipais que podem enriquecer as análises exploratórias e futuras etapas de modelagem.



### Carrega dados ml-alunos utilizando os passos do TC3_38

In [ ]:
import polars as pl
import pandas as pd
import zipfile
from pathlib import Path

In [ ]:

# Localiza a raiz do projeto a partir do diretório atual do notebook.
# Assim o notebook funciona no VS Code, Jupyter local ou outro clone do repositório.
def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "dados" / "principal" / "ml_aluno.zip").exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível encontrar dados/principal/ml_aluno.zip. "
        "Execute o notebook dentro do repositório ou confira se os dados foram baixados."
    )

PROJECT_ROOT = encontrar_raiz_projeto()
ZIP_PATH = PROJECT_ROOT / "dados" / "principal" / "ml_aluno.zip"
EXTRACT_PATH = PROJECT_ROOT / "dados" / "principal"

if not (EXTRACT_PATH / "ml_aluno").exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

# Caminho local da feature table extraída
BASE_PATH = EXTRACT_PATH / "ml_aluno"

# Partições da Gold
PATH_2023 = BASE_PATH / "ano2023"
PATH_2024 = BASE_PATH / "ano2024"

arquivos_2023 = sorted(PATH_2023.glob("*.parquet"))
arquivos_2024 = sorted(PATH_2024.glob("*.parquet"))

df_2023 = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos_2023],
    ignore_index=True
)

df_2024 = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos_2024],
    ignore_index=True
)

# Consolidação da base
df = pd.concat([df_2023, df_2024], ignore_index=True)



### Carrega arquivos complementares

In [ ]:


PATH_COMPLEMENTAR = PROJECT_ROOT / "dados" / "complementar" 

#Bolsa Familia
df_bolsa_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_dados_bolsa_familia_2023_2024.csv")

#Fundeb
df_fundeb_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_dados_fundeb_indicador_municipal_2023_2024.csv")

#Receitas Orcamentarias
df_receitas_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_de_dados_municipio_receitas_orcamentarias_2023_2024.csv")

#PIB
df_pib_bronze = pd.read_csv(
    PATH_COMPLEMENTAR / "base_de_dados_PIB_Municipios_2022_2023.csv",
    sep=";",           # o arquivo usa ;
    decimal=","        # números com vírgula
)

#INSE
df_inse_bronze = pd.read_excel(PATH_COMPLEMENTAR / "INSE_2023_escolas.xlsx", sheet_name="INSE_ESC_2023")  # ou sheet_name=0 (primeira aba)


#Censo Escolar
ZIP_PATH_CENSO = PATH_COMPLEMENTAR / "microdados_ed_basica_2023.zip"
CSV_PATH_CENSO = PATH_COMPLEMENTAR / "microdados_ed_basica_2023.csv"

if ZIP_PATH_CENSO.exists() and not CSV_PATH_CENSO.exists():
    with zipfile.ZipFile(ZIP_PATH_CENSO, "r") as zip_ref:
        zip_ref.extractall(PATH_COMPLEMENTAR)

if CSV_PATH_CENSO.exists():
    df_censo_bronze = pd.read_csv(
        CSV_PATH_CENSO,
        sep=";",
        decimal=",",
        encoding="latin1"
    )
else:
    print("Arquivo do censo não encontrado.")








### Padronizar os nomes das colunas

In [ ]:
# Bolsa Família
df_bolsa_silver = (
    df_bolsa_bronze
    .rename(columns={
        "ano_referencia": "ano",
        "total_valor": "valor_bolsa_familia"
    })
    .rename(columns=lambda col: col.lower())
)

# Fundeb
df_fundeb_silver = (
    df_fundeb_bronze
    .rename(columns={
        "total_valor": "valor_fundeb"
    })
    .rename(columns=lambda col: col.lower())
)

# Receitas orçamentárias
df_receitas_silver = (
    df_receitas_bronze
    .rename(columns={
        "total_valor": "valor_receitas"
    })
    .rename(columns=lambda col: col.lower())
)

# PIB
df_pib_silver = (
    df_pib_bronze
    .rename(columns={
        "Ano": "ano",
        "UF": "sigla_uf",
        "Código do Município": "id_municipio",
        "Produto Interno Bruto, a preços correntes": "valor_pib",
        "Produto Interno Bruto per capita, a preços correntes": "valor_pib_per_capita"
    })
    .rename(columns=lambda col: col.lower())
)

# INSE
df_inse_silver = (
    df_inse_bronze
    .rename(columns={
        "NU_ANO_SAEB": "ano",
        "SG_UF": "sigla_uf",
        "NO_UF": "nome_uf",
        "CO_MUNICIPIO": "id_municipio",
        "NO_MUNICIPIO": "nome_municipio",
        "ID_ESCOLA": "id_escola",
        "NO_ESCOLA": "nome_escola",
        "TP_TIPO_REDE": "tipo_rede",
        "TP_LOCALIZACAO": "localizacao",
        "TP_CAPITAL": "capital",
        "QTD_ALUNOS_INSE": "qtd_alunos_inse",
        "MEDIA_INSE": "media_inse",
        "INSE_CLASSIFICACAO": "inse_classificacao",
        "PC_NIVEL_1": "pc_nivel_1",
        "PC_NIVEL_2": "pc_nivel_2",
        "PC_NIVEL_3": "pc_nivel_3",
        "PC_NIVEL_4": "pc_nivel_4",
        "PC_NIVEL_5": "pc_nivel_5",
        "PC_NIVEL_6": "pc_nivel_6",
        "PC_NIVEL_7": "pc_nivel_7",
        "PC_NIVEL_8": "pc_nivel_8"
    })
    .rename(columns=lambda col: col.lower())
)

# Censo
df_censo_silver = (
    df_censo_bronze
    .rename(columns={
        "NU_ANO_CENSO": "ano",
        "SG_UF": "sigla_uf",
        "NO_UF": "nome_uf",
        "CO_UF": "co_uf",
        "CO_MUNICIPIO": "id_municipio",
        "NO_MUNICIPIO": "nome_municipio",
        "CO_ENTIDADE": "id_escola",
        "NO_ENTIDADE": "nome_escola",
        "NO_REGIAO": "nome_regiao",
        "CO_REGIAO": "co_regiao",
        "TP_LOCALIZACAO": "localizacao",
        "TP_LOCALIZACAO_DIFERENCIADA": "localizacao_diferenciada",
        "TP_DEPENDENCIA": "dependencia",
        "IN_BIBLIOTECA_SALA_LEITURA": "tem_biblioteca",
        "IN_LABORATORIO_INFORMATICA": "tem_lab_informatica",
        "IN_QUADRA_ESPORTES": "tem_quadra",
        "IN_AGUA_REDE_PUBLICA": "tem_agua_rede_publica",
        "IN_ENERGIA_REDE_PUBLICA": "tem_energia_rede_publica",
        "IN_ESGOTO_REDE_PUBLICA": "tem_esgoto_rede_publica",
        "TP_SITUACAO_FUNCIONAMENTO": "situacao_funcionamento",
        "IN_INTERNET": "tem_internet",
        "IN_BANDA_LARGA": "tem_banda_larga",
        "QT_MAT_BAS": "qtd_matriculas_basica",
        "QT_DOC_BAS": "qtd_docentes_basica",
        "QT_TUR_BAS": "qtd_turmas_basica"
    })
    .rename(columns=lambda col: col.lower())
)


### Padronizar os tipos das colunas

In [ ]:
df["id_municipio"] = df["id_municipio"].astype("int64")
df["ano_referencia"] = df["ano_referencia"].astype("int64")

df_bolsa_silver["valor_bolsa_familia"] = df_bolsa_silver["valor_bolsa_familia"].astype("float64")
df_fundeb_silver["valor_fundeb"] = df_fundeb_silver["valor_fundeb"].astype("float64")
df_receitas_silver["valor_receitas"] = df_receitas_silver["valor_receitas"].astype("float64")
df_pib_silver["valor_pib"] = (
    df_pib_silver["valor_pib"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)
df_pib_silver["valor_pib_per_capita"] = (
    df_pib_silver["valor_pib_per_capita"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

colunas_com_codigo = [
    coluna
    for coluna in df_censo_silver.columns
    if df_censo_silver[coluna].astype("string").eq("0MI11").any()
]

for coluna in colunas_com_codigo:
    df_censo_silver[coluna] = df_censo_silver[coluna].astype("string")

### Unir todas as bases

#### Todas as fontes são consolidadas no nível município/ano antes do `join`. INSE e Censo chegam no nível escolar e são agregados para proteger a granularidade adotada e impedir a explosão de linhas.

In [ ]:

df_aluno_silver = pl.DataFrame(df)
df_bolsa_silver = pl.DataFrame(df_bolsa_silver)  # Converte de Pandas para Polars
df_fundeb_silver = pl.DataFrame(df_fundeb_silver)  # Converte de Pandas para Polars
df_receitas_silver = pl.DataFrame(df_receitas_silver)  # Converte de Pandas para Polars
df_pib_silver = pl.DataFrame(df_pib_silver)  # Converte de Pandas para Polars
df_inse_silver = pl.DataFrame(df_inse_silver)  # Converte de Pandas para Polars
df_censo_silver = pl.DataFrame(df_censo_silver)  # Converte de Pandas para Polars 

chaves_municipio = ["ano", "id_municipio", "sigla_uf"]

# O INSE é escolar na origem. A média e os percentuais são ponderados
# pela quantidade de alunos representados em cada escola.
colunas_inse_ponderadas = [
    "media_inse",
    "pc_nivel_1", "pc_nivel_2", "pc_nivel_3", "pc_nivel_4",
    "pc_nivel_5", "pc_nivel_6", "pc_nivel_7", "pc_nivel_8",
]
df_inse_municipal = (
    df_inse_silver
    .with_columns(
        pl.col("qtd_alunos_inse").cast(pl.Float64, strict=False),
        *[pl.col(col).cast(pl.Float64, strict=False) for col in colunas_inse_ponderadas],
    )
    .group_by(chaves_municipio)
    .agg(
        pl.col("id_escola").n_unique().alias("qtd_escolas_inse"),
        pl.col("qtd_alunos_inse").sum().alias("qtd_alunos_inse"),
        *[
            (
                (pl.col(col) * pl.col("qtd_alunos_inse")).sum()
                / pl.col("qtd_alunos_inse").sum()
            ).alias(col)
            for col in colunas_inse_ponderadas
        ],
    )
)

# O Censo também é escolar na origem. Totais são somados e indicadores
# binários viram a proporção de escolas do município com cada recurso.
colunas_censo_totais = [
    "qtd_matriculas_basica", "qtd_docentes_basica", "qtd_turmas_basica",
]
colunas_censo_indicadores = [
    "tem_biblioteca", "tem_lab_informatica", "tem_quadra",
    "tem_agua_rede_publica", "tem_energia_rede_publica",
    "tem_esgoto_rede_publica", "tem_internet", "tem_banda_larga",
]
df_censo_municipal = (
    df_censo_silver
    .with_columns(
        *[
            pl.col(col).cast(pl.Float64, strict=False)
            for col in colunas_censo_totais + colunas_censo_indicadores
        ]
    )
    .group_by(chaves_municipio)
    .agg(
        pl.col("id_escola").n_unique().alias("qtd_escolas_censo"),
        *[pl.col(col).sum().alias(col) for col in colunas_censo_totais],
        *[
            pl.col(col).mean().alias(f"proporcao_escolas_{col.removeprefix('tem_')}")
            for col in colunas_censo_indicadores
        ],
    )
)

qtd_linhas_antes_dos_joins = df_aluno_silver.height

# Junta Bolsa Família
df_aluno_silver = df_aluno_silver.join(
    df_bolsa_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    validate="m:1"
)

# Junta Fundeb
df_aluno_silver = df_aluno_silver.join(
    df_fundeb_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    validate="m:1"
)

# Junta Receitas
df_aluno_silver = df_aluno_silver.join(
    df_receitas_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    validate="m:1"
)

df_aluno_silver = df_aluno_silver.join(
    df_pib_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    validate="m:1"
)

# Junta INSE já consolidado por município
df_aluno_silver = df_aluno_silver.join(
    df_inse_municipal,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=chaves_municipio,
    validate="m:1"
)

# Junta Censo Escolar já consolidado por município
df_aluno_silver = df_aluno_silver.join(
    df_censo_municipal,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=chaves_municipio,
    validate="m:1"
)

assert df_aluno_silver.height == qtd_linhas_antes_dos_joins, (
    "A quantidade de alunos mudou durante os joins: "
    f"{qtd_linhas_antes_dos_joins:,} -> {df_aluno_silver.height:,}"
)


In [ ]:
print(f"Shape: {df_aluno_silver.shape}")
df_aluno_silver.glimpse()
df_aluno_silver.head()

### Salvar a base consolidada

O dataset abaixo mantém uma linha por aluno, reúne as variáveis municipais disponíveis para o respectivo ano e grava os arquivos Parquet em partições por UF.

In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT / "dados" / "consolidado" / "base_alunos_municipios_por_uf"
)

df_aluno_silver.write_parquet(
    OUTPUT_DIR,
    compression="zstd",
    statistics=True,
    partition_by="sigla_uf",
    mkdir=True,
)

arquivos_parquet = list(OUTPUT_DIR.rglob("*.parquet"))
tamanho_mb = sum(arq.stat().st_size for arq in arquivos_parquet) / (1024 ** 2)
print(f"Base consolidada particionada salva em: {OUTPUT_DIR}")
print(f"Linhas: {df_aluno_silver.height:,} | Colunas: {df_aluno_silver.width:,}")
print(f"Partições/arquivos: {len(arquivos_parquet)}")
print(f"Tamanho: {tamanho_mb:,.2f} MB")